In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('train.csv')
df_full = pd.read_excel('US_dataQ.xlsx')
cols = ['FEDFUNDS', 'DTWEXBGS', 'GDPC1', 'CPIAUCSL', 'RTWEXBGS']

def transform(frame):
    out = frame.assign(Date=pd.to_datetime(frame['Date'])).set_index('Date')[cols].astype(float).sort_index()
    out['FEDFUNDS'] = out['FEDFUNDS'].diff()
    out[cols[1:]] = np.log(out[cols[1:]]).diff()
    return out.dropna()

data, data_full = transform(df), transform(df_full)
train_end = data.index.max()

In [6]:
data

,FEDFUNDS,DTWEXBGS,GDPC1,CPIAUCSL,RTWEXBGS
Date,,,,,
2006-06-30,0.450000,-0.016280,0.002584,0.008984,-0.012164
2006-09-30,0.340000,-0.005734,0.001498,0.009396,-0.000158
2006-12-31,0.000000,-0.004925,0.008558,-0.004110,-0.015062
2007-03-31,0.010000,-0.001162,0.003004,0.009756,0.000804
2007-06-30,-0.006667,-0.026127,0.006099,0.011262,-0.021359
...,...,...,...,...,...
2023-03-31,0.863333,-0.036747,0.007212,0.008966,-0.035978
2023-06-30,0.473333,-0.007618,0.006259,0.007169,-0.008094
2023-09-30,0.270000,0.003796,0.011470,0.008505,0.001854


## PySR для инфляции с residual моделей FEDFUNDS

Для каждой модели ставки и горизонта строится исторический residual на дате $t$:

$$
u^{(m,h)}_t
=
FEDFUNDS_t
-
\left(
FEDFUNDS_{t-h}
+
\sum_{k=1}^{h}\widehat{\Delta FEDFUNDS}^{(m,k)}_{t-h+k\mid t-h}
\right).
$$

Таким образом, фактический уровень и прогноз сравниваются на одной дате, а прогноз был сделан за $h$ кварталов до неё. Для прогноза инфляции используются текущие значения и лаги 1–2 рядов DTWEXBGS, GDPC1, RTWEXBGS и этого residual.


In [7]:
import os
import json
import cloudpickle
from pathlib import Path
from pysr import PySRRegressor

RATE_MODEL_DIR = Path('models/fedfunds')
PYSR_MODEL_DIR = Path(os.getenv('PYSR_MODEL_DIR', 'models/inflation_pysr'))
PYSR_RUN_DIR = PYSR_MODEL_DIR / 'runs'
PYSR_MODEL_DIR.mkdir(parents=True, exist_ok=True)
PYSR_RUN_DIR.mkdir(parents=True, exist_ok=True)

PYSR_NITERATIONS = int(os.getenv('PYSR_NITERATIONS', 100))
PYSR_LIMIT = int(os.getenv('PYSR_LIMIT', 0)) or None
REFIT_PYSR = os.getenv('REFIT_PYSR', '0') == '1'

RATE_MODEL_NAMES = sorted(p.stem.rsplit('_fedfunds_h', 1)[0]
                          for p in RATE_MODEL_DIR.glob('*_fedfunds_h1.pkl')
                          if not p.stem.startswith('tuned_params_'))
assert RATE_MODEL_NAMES and all((RATE_MODEL_DIR / f'{name}_fedfunds_h{h}.pkl').exists()
                                for name in RATE_MODEL_NAMES for h in range(1, 9))
len(RATE_MODEL_NAMES), RATE_MODEL_NAMES


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


(22,
 ['ard',
  'bayesian_ridge',
  'catboost',
  'catboost_v2',
  'decision_tree',
  'decision_tree_v2',
  'elastic_net',
  'gaussian_process',
  'glsar',
  'huber',
  'kernel_ridge',
  'knn',
  'lasso',
  'lgbm',
  'lgbm_v2',
  'random_forest',
  'random_forest_v2',
  'ransac',
  'ridge',
  'svr',
  'xgboost',
  'xgboost_v2'])

### Признаки для сохранённых моделей ставки

Функция ниже полностью повторяет построение 11 входных признаков из rate_modls.ipynb, поэтому порядок и смысл столбцов совпадают с обученными pkl.


In [8]:
def make_features(z, h):
    F1, F2, F3 = (z.FEDFUNDS.shift(i) for i in (1, 2, 3))
    D1, D2, D3 = (z.DTWEXBGS.shift(i) for i in (1, 2, 3))
    G1, G2, G3 = (z.GDPC1.shift(i) for i in (1, 2, 3))
    C1, C2, C3 = (z.CPIAUCSL.shift(i) for i in (1, 2, 3))
    R1, R2, R3 = (z.RTWEXBGS.shift(i) for i in (1, 2, 3))

    engineered = {
        1: lambda: [
            np.sin(F1 * (1.5105 - F3)),
            1.4905 * np.sin(F1 * (F3 - 1.0672)**2),
            1.545 * np.sin(F1 * (F3 + D2 - 1.0757)**2),
            1.6005 * np.sin(F1 * (F3 - 1.0543 - 3.8208 * D2 * F2)**2),
            (1.0085 + C1 / .02507) * np.sin(F1 * (1 + (F3 - .54019)**2 - F3)),
            (.99788 + C3 / .025327 - D2 / .081413) * np.sin(F1 * (1 + (F3 - .51752)**2 - F3))
        ],
        2: lambda: [
            6.0423 * R1 + np.sin(np.sin(F1)),
            np.sin(F1) + .4145 * np.sin(12.67 * F1) - G1 / .11288,
            .41701 * np.sin(12.597 * (F1 + G1)) + np.sin(F1) - G1 / .12789,
            np.sin(1.8377 * F1) + .39022 * np.sin(13.492 * (F1 + C3)) - .33503 * F3,
            .38827 * np.sin(13.486 * (F1 + C2 + C3)) - .33506 * F3 + np.sin(1.8505 * F1),
            .38932 * np.sin(13.194 * (F1 + 2.867 * C3)) + np.sin(1.7363 * F1) - .31582 * F3 - G1 / .12759
        ],
        3: lambda: [
            C1**2 * (C1 + R2),
            (174.68 * C1 - 2.6608 * F1)**2 * (R2 + C1),
            (R1 / R2 + 1.5141) * (R2 + C1) * (174.68 * C1 - 2.6608 * F1)**2,
            np.sin((22.525 * C1**2 / D2 + .28412)**2 - .11499 + F1 / 4.0387),
            1.4565 * np.sin((22.545 * C1**2 / D2 + .28325)**2 - .11479 + F1 / 3.9611 - C2),
            (1.631 + R1 / R2) * ((R2 + C1) * (174.68 * C1 - 2.6554 * F1)**2 - R2) - G1
        ],
        4: lambda: [
            22.8 * C1 * C2 / R1,
            np.sin(27.226 * C1 * C2 / R1),
            np.sin(16.841 * C1 * C2 * C3 / (D2 * R3)),
            (C1 / D2 - F1) * np.sin(20.196 * C3 * (C1 / R3 - .75193)),
            np.sin((R1 / R2 + 3.4861 * C1 / R1) * (595.71 * C1 * C2 + D1)),
            1.243 * np.sin(563.8 * C1 * C2 * (R1 / R2 + C1 * (3.2654 / R1 + 1.1972 / R2)))
        ],
        5: lambda: [
            np.sin(5923.5**2 * G3**2 * C1**2),
            np.sin((3.1106 * C1 / R3 - 1.695)**2 * (G3 + G2)),
            np.sin(((2.9252 * C1 / R3 - 1.7253)**2 - R1 / R2) * (G1 + G2)),
            1.5248 * np.sin(((2.9252 * C1 / R3 - 1.7253)**2 + C3 / (.61061 * R2)) * (G1 + G2)),
            1.3843 * np.sin(G2 / R2 * ((.60449 * C1 / D1)**2 - .065813)),
            (1.719 - F1**2) * np.sin(C1 / R2 * np.sin(.28364 * C1**2 / D1**2 - .049407))
        ],
        6: lambda: [
            (D1 / R1 - 1.2875)**2 * (50.549 * G3)**2,
            1.4863 * np.sin((52.583 * G3 * (D1 / R1 - 1.2993))**2),
            (15.327 * (R1 / D1 - .68966) * (R1 + 3.2356 * G3))**2 - .085454 * (R1 / D1)**2,
            (G1 + 2.4972 * C1) * (3.7732 + C2 / R1 - .021685 / R2) + G1,
            G1 * C1 / R1 * (5.0829 + C2 / R2) * (1.8381 - 3.0663 * C2 / D2),
            (4.9792 + C2 / R2) * (G1 + C1 / R1 * (R1 + G1 * (2.021 - 3.2115 * C2 / D2))) + G2
        ],
        7: lambda: [
            G2 / np.sin(4.4741 / C1),
            2.6389 * (C1 / np.sin(-.049636 / (R2 * G1)) + G2 / np.sin(3.7701 / C1) + C1 * C2 / R1),
            (G3 / .065448)**2 + G1 / .23016 - .0033526 * F3 / R1 + R3,
            (F3 * R2 / (28.84 * R1))**2 + (D3 + G3 * (F2 - .58751) / -.035273)**2,
            (F3 * R2 / (28.84 * R1))**2 + (2 * D3 + G3 * (F2 - .59881) / -.035622)**2 + G1 + R3,
            (F3 * R2 / (28.84 * R1))**2 + (G3 * (F2 - .59881) / -.035273 + G2 + 2 * D3)**2 + G1 + C1 + R3
        ],
        8: lambda: [
            (12.255 * (G1 - G3))**2,
            (G3 - G1)**2 * (R2 / C2 - 10.571)**2,
            (12.367 * (G1 - G3))**2 - (F3 * (F3 + 1.2667))**2,
            233.23 * G2**2 + G1 / D1 * (3.1726 * D2 + G1 * R1 / R2),
            (np.sin(F1) + G1 / D1) * (G1 * R1 / R2 + 3.3905 * D2) + 235.35 * G2**2 + R2,
            237.65 * G2**2 + 2 * R2 + G1 / D1 * (3.6347 * D2 + R1 * G1 / R2) - .20974 * np.sin(np.sin(F1))
        ]
    }
    X = z[cols].copy()
    for i, feature in enumerate(engineered[h](), 1):
        X[f'formula_{i}'] = feature
    return X.replace([np.inf, -np.inf], np.nan)


In [9]:
fedfunds_level = df.assign(Date=pd.to_datetime(df.Date)).set_index('Date').FEDFUNDS.astype(float)
rate_features = {h: make_features(data, h) for h in range(1, 9)}
rate_delta_predictions = {}

for name in RATE_MODEL_NAMES:
    for h in range(1, 9):
        with (RATE_MODEL_DIR / f'{name}_fedfunds_h{h}.pkl').open('rb') as file:
            rate_model = cloudpickle.load(file)
        valid = rate_features[h].dropna().index
        prediction = pd.Series(np.nan, index=data.index, name=f'{name}_h{h}')
        prediction.loc[valid] = rate_model.predict(rate_features[h].loc[valid])
        rate_delta_predictions[name, h] = prediction


def make_fedfunds_residual(name, h):
    predicted_diffs = pd.concat([rate_delta_predictions[name, k] for k in range(1, h + 1)], axis=1).dropna()
    positions = data.index.get_indexer(predicted_diffs.index)
    predicted_diffs = predicted_diffs.iloc[positions + h < len(data)]
    positions = data.index.get_indexer(predicted_diffs.index)
    target_dates = data.index[positions + h]
    predicted_level = fedfunds_level.loc[predicted_diffs.index].to_numpy() + predicted_diffs.sum(axis=1).to_numpy()
    return pd.Series(fedfunds_level.loc[target_dates].to_numpy() - predicted_level,
                     index=target_dates, name='FEDFUNDS_GAP')


fedfunds_residuals = {(h, name): make_fedfunds_residual(name, h)
                      for h in range(1, 9) for name in RATE_MODEL_NAMES}


### Выборки CPIAUCSL

PySR получает 12 кандидатов: четыре ряда на $t$, $t-1$ и $t-2$. Параметр select_k_features=6 оставляет не более шести признаков для каждого сочетания горизонта CPI и модели ставки.


In [10]:
MACRO_FEATURES = ['DTWEXBGS', 'GDPC1', 'RTWEXBGS']


def inflation_xy(h, rate_name):
    source = data[MACRO_FEATURES].copy()
    source['FEDFUNDS_GAP'] = fedfunds_residuals[h, rate_name]
    features, definitions = {}, {}

    for name in source:
        for lag in range(3):
            feature_name = f'{name}_t' if lag == 0 else f'{name}_lag{lag}'
            features[feature_name] = source[name].shift(lag)
            base = f'Δlog({name})' if name in MACRO_FEATURES else f'FEDFUNDS − forecast_level({rate_name}, h={h})'
            definitions[feature_name] = f'{base} at t' if lag == 0 else f'{base} at t−{lag}'

    y = data.CPIAUCSL.shift(-h).rename(f'CPIAUCSL_h{h}')
    sample = pd.DataFrame(features).join(y).dropna()
    return sample.drop(columns=y.name), sample[y.name], definitions


inflation_samples = {(h, name): inflation_xy(h, name)
                     for h in range(1, 9) for name in RATE_MODEL_NAMES}
pd.DataFrame({'horizon': h, 'rate_model': name, 'n': len(inflation_samples[h, name][1]),
              'p': inflation_samples[h, name][0].shape[1]}
             for h in range(1, 9) for name in RATE_MODEL_NAMES)


,horizon,rate_model,n,p
0,1,ard,65,12
1,1,bayesian_ridge,65,12
2,1,catboost,65,12
3,1,catboost_v2,65,12
4,1,decision_tree,65,12
...,...,...,...,...
171,8,ransac,51,12
172,8,ridge,51,12
173,8,svr,51,12
174,8,xgboost,51,12


### PySR по всем горизонтам и моделям ставки

Каждая комбинация сохраняется сразу после завершения, поэтому ячейку можно безопасно перезапускать: готовые pkl пропускаются. Переменные окружения PYSR_LIMIT и PYSR_NITERATIONS позволяют сделать короткий проверочный запуск; REFIT_PYSR=1 принудительно пересчитывает существующие модели.


In [11]:
DICTIONARY_PKL = PYSR_MODEL_DIR / 'inflation_feature_dictionary.pkl'
DICTIONARY_JSON = PYSR_MODEL_DIR / 'inflation_feature_dictionary.json'

if DICTIONARY_PKL.exists() and not REFIT_PYSR:
    with DICTIONARY_PKL.open('rb') as file:
        feature_dictionary = cloudpickle.load(file)
else:
    feature_dictionary = {}


def pysr_path(h, rate_name):
    return PYSR_MODEL_DIR / f'pysr_cpiaucsl_h{h}_{rate_name}_fedfunds.pkl'


def load_pysr(h, rate_name):
    with pysr_path(h, rate_name).open('rb') as file:
        return cloudpickle.load(file)


def feature_record(model, h, rate_name, X, y, definitions):
    best = model.get_best()
    mask = model.selection_mask_
    selected = X.columns.tolist() if mask is None else X.columns[np.asarray(mask)].tolist()
    expression = best['sympy_format']
    used = sorted(str(symbol) for symbol in expression.free_symbols)
    return {
        'target': f'Δlog(CPIAUCSL)_t+{h}',
        'fedfunds_residual': f'FEDFUNDS_t - forecast_level_{rate_name}_made_{h}q_earlier',
        'candidate_features': X.columns.tolist(),
        'selected_features': selected,
        'used_features': used,
        'selected_transformations': {name: definitions[name] for name in selected},
        'used_transformations': {name: definitions[name] for name in used},
        'equation': str(best['equation']),
        'sympy': str(expression),
        'train_mse': float(best['loss']),
        'complexity': int(best['complexity']),
        'n_observations': len(y),
        'model_pkl': str(pysr_path(h, rate_name))
    }


def save_feature_dictionary():
    with DICTIONARY_PKL.open('wb') as file:
        cloudpickle.dump(feature_dictionary, file)
    with DICTIONARY_JSON.open('w', encoding='utf-8') as file:
        json.dump(feature_dictionary, file, ensure_ascii=False, indent=2)


combinations = [(h, name) for h in range(1, 9) for name in RATE_MODEL_NAMES]
combinations = combinations[:PYSR_LIMIT] if PYSR_LIMIT else combinations
for number, (h, rate_name) in enumerate(combinations, 1):
    X, y, definitions = inflation_samples[h, rate_name]
    path = pysr_path(h, rate_name)

    if path.exists() and not REFIT_PYSR:
        model = load_pysr(h, rate_name)
    else:
        model = PySRRegressor(
            niterations=PYSR_NITERATIONS,
            binary_operators=['+', '-', '*', '/'],
            unary_operators=['square', 'sqrt', 'log', 'sin'],
            nested_constraints={'/': {'+': 0, '-': 0}},
            elementwise_loss='loss(prediction, target) = (prediction - target)^2',
            select_k_features=min(6, X.shape[1]),
            maxsize=20,
            model_selection='best',
            output_directory=str(PYSR_RUN_DIR),
            random_state=42,
            parallelism='serial',
            deterministic=True,
            progress=False,
            verbosity=0
        ).fit(X, y)
        with path.open('wb') as file:
            cloudpickle.dump(model, file)

    feature_dictionary.setdefault(f'h{h}', {})[rate_name] = feature_record(
        model, h, rate_name, X, y, definitions
    )
    save_feature_dictionary()
    print(f'{number}/{len(combinations)}: h={h}, FEDFUNDS={rate_name}')


1/176: h=1, FEDFUNDS=ard
2/176: h=1, FEDFUNDS=bayesian_ridge
3/176: h=1, FEDFUNDS=catboost
4/176: h=1, FEDFUNDS=catboost_v2
5/176: h=1, FEDFUNDS=decision_tree
6/176: h=1, FEDFUNDS=decision_tree_v2
7/176: h=1, FEDFUNDS=elastic_net
8/176: h=1, FEDFUNDS=gaussian_process
9/176: h=1, FEDFUNDS=glsar
10/176: h=1, FEDFUNDS=huber
11/176: h=1, FEDFUNDS=kernel_ridge
12/176: h=1, FEDFUNDS=knn
13/176: h=1, FEDFUNDS=lasso
14/176: h=1, FEDFUNDS=lgbm
15/176: h=1, FEDFUNDS=lgbm_v2
16/176: h=1, FEDFUNDS=random_forest
17/176: h=1, FEDFUNDS=random_forest_v2
18/176: h=1, FEDFUNDS=ransac
19/176: h=1, FEDFUNDS=ridge
20/176: h=1, FEDFUNDS=svr
21/176: h=1, FEDFUNDS=xgboost
22/176: h=1, FEDFUNDS=xgboost_v2
23/176: h=2, FEDFUNDS=ard
24/176: h=2, FEDFUNDS=bayesian_ridge
25/176: h=2, FEDFUNDS=catboost
26/176: h=2, FEDFUNDS=catboost_v2
27/176: h=2, FEDFUNDS=decision_tree
28/176: h=2, FEDFUNDS=decision_tree_v2
29/176: h=2, FEDFUNDS=elastic_net
30/176: h=2, FEDFUNDS=gaussian_process
31/176: h=2, FEDFUNDS=glsar
32/176

19/176: h=1, FEDFUNDS=ridge
20/176: h=1, FEDFUNDS=svr
21/176: h=1, FEDFUNDS=xgboost
22/176: h=1, FEDFUNDS=xgboost_v2
23/176: h=2, FEDFUNDS=ard
24/176: h=2, FEDFUNDS=bayesian_ridge
25/176: h=2, FEDFUNDS=catboost
26/176: h=2, FEDFUNDS=catboost_v2
27/176: h=2, FEDFUNDS=decision_tree
28/176: h=2, FEDFUNDS=decision_tree_v2


29/176: h=2, FEDFUNDS=elastic_net
30/176: h=2, FEDFUNDS=gaussian_process
31/176: h=2, FEDFUNDS=glsar
32/176: h=2, FEDFUNDS=huber
33/176: h=2, FEDFUNDS=kernel_ridge
34/176: h=2, FEDFUNDS=knn
35/176: h=2, FEDFUNDS=lasso


36/176: h=2, FEDFUNDS=lgbm
37/176: h=2, FEDFUNDS=lgbm_v2
38/176: h=2, FEDFUNDS=random_forest
39/176: h=2, FEDFUNDS=random_forest_v2
40/176: h=2, FEDFUNDS=ransac
41/176: h=2, FEDFUNDS=ridge
42/176: h=2, FEDFUNDS=svr
43/176: h=2, FEDFUNDS=xgboost
44/176: h=2, FEDFUNDS=xgboost_v2
45/176: h=3, FEDFUNDS=ard
46/176: h=3, FEDFUNDS=bayesian_ridge
47/176: h=3, FEDFUNDS=catboost


48/176: h=3, FEDFUNDS=catboost_v2
49/176: h=3, FEDFUNDS=decision_tree
50/176: h=3, FEDFUNDS=decision_tree_v2
51/176: h=3, FEDFUNDS=elastic_net
52/176: h=3, FEDFUNDS=gaussian_process
53/176: h=3, FEDFUNDS=glsar
54/176: h=3, FEDFUNDS=huber
55/176: h=3, FEDFUNDS=kernel_ridge
56/176: h=3, FEDFUNDS=knn
57/176: h=3, FEDFUNDS=lasso
58/176: h=3, FEDFUNDS=lgbm
59/176: h=3, FEDFUNDS=lgbm_v2
60/176: h=3, FEDFUNDS=random_forest
61/176: h=3, FEDFUNDS=random_forest_v2
62/176: h=3, FEDFUNDS=ransac


...

139/176: h=7, FEDFUNDS=elastic_net
140/176: h=7, FEDFUNDS=gaussian_process
141/176: h=7, FEDFUNDS=glsar
142/176: h=7, FEDFUNDS=huber
143/176: h=7, FEDFUNDS=kernel_ridge
144/176: h=7, FEDFUNDS=knn
145/176: h=7, FEDFUNDS=lasso
146/176: h=7, FEDFUNDS=lgbm
147/176: h=7, FEDFUNDS=lgbm_v2
148/176: h=7, FEDFUNDS=random_forest


149/176: h=7, FEDFUNDS=random_forest_v2
150/176: h=7, FEDFUNDS=ransac
151/176: h=7, FEDFUNDS=ridge
152/176: h=7, FEDFUNDS=svr
153/176: h=7, FEDFUNDS=xgboost


154/176: h=7, FEDFUNDS=xgboost_v2
155/176: h=8, FEDFUNDS=ard
156/176: h=8, FEDFUNDS=bayesian_ridge
157/176: h=8, FEDFUNDS=catboost
158/176: h=8, FEDFUNDS=catboost_v2
159/176: h=8, FEDFUNDS=decision_tree
160/176: h=8, FEDFUNDS=decision_tree_v2


161/176: h=8, FEDFUNDS=elastic_net
162/176: h=8, FEDFUNDS=gaussian_process
163/176: h=8, FEDFUNDS=glsar
164/176: h=8, FEDFUNDS=huber
165/176: h=8, FEDFUNDS=kernel_ridge
166/176: h=8, FEDFUNDS=knn
167/176: h=8, FEDFUNDS=lasso
168/176: h=8, FEDFUNDS=lgbm
169/176: h=8, FEDFUNDS=lgbm_v2


170/176: h=8, FEDFUNDS=random_forest
171/176: h=8, FEDFUNDS=random_forest_v2
172/176: h=8, FEDFUNDS=ransac
173/176: h=8, FEDFUNDS=ridge
174/176: h=8, FEDFUNDS=svr
175/176: h=8, FEDFUNDS=xgboost
176/176: h=8, FEDFUNDS=xgboost_v2


In [12]:
feature_summary = pd.DataFrame(
    {'horizon': int(h[1:]), 'rate_model': name, 'train_mse': values.get('train_mse', values.get('mse')),
     'features': ', '.join(values['used_features']), 'equation': values['equation'],
     'pkl': values['model_pkl']}
    for h, models in feature_dictionary.items() for name, values in models.items()
).sort_values(['horizon', 'train_mse']).reset_index(drop=True)

feature_summary


,horizon,rate_model,train_mse,features,equation,pkl
0,1,ransac,0.000024,"DTWEXBGS_t, FEDFUNDS_GAP_lag2, GDPC1_lag2, RTW...",(sin((0.7758381 - FEDFUNDS_GAP_lag2) * FEDFUND...,models/inflation_pysr/pysr_cpiaucsl_h1_ransac_...
1,1,bayesian_ridge,0.000031,"DTWEXBGS_t, FEDFUNDS_GAP_lag2, FEDFUNDS_GAP_t,...",(GDPC1_lag2 + 0.055891905) * (0.10886703 - (DT...,models/inflation_pysr/pysr_cpiaucsl_h1_bayesia...
2,1,lasso,0.000033,"FEDFUNDS_GAP_lag2, FEDFUNDS_GAP_t, GDPC1_lag2,...",square((((((GDPC1_lag2 - FEDFUNDS_GAP_t) * (FE...,models/inflation_pysr/pysr_cpiaucsl_h1_lasso_f...
3,1,lgbm_v2,0.000035,"DTWEXBGS_t, FEDFUNDS_GAP_lag2",0.00720308 - (((FEDFUNDS_GAP_lag2 - 0.26076645...,models/inflation_pysr/pysr_cpiaucsl_h1_lgbm_v2...
4,1,ridge,0.000036,"FEDFUNDS_GAP_lag2, GDPC1_lag2, GDPC1_t",sqrt(square(((GDPC1_t + 0.011183656) + GDPC1_l...,models/inflation_pysr/pysr_cpiaucsl_h1_ridge_f...
...,...,...,...,...,...,...
171,8,decision_tree,0.000027,"FEDFUNDS_GAP_lag1, GDPC1_t",square(square(FEDFUNDS_GAP_lag1 - -1.257136) *...,models/inflation_pysr/pysr_cpiaucsl_h8_decisio...
172,8,lgbm,0.000027,"GDPC1_t, RTWEXBGS_lag1",(square(GDPC1_t / RTWEXBGS_lag1) * 0.000183900...,models/inflation_pysr/pysr_cpiaucsl_h8_lgbm_fe...
173,8,kernel_ridge,0.000027,"GDPC1_t, RTWEXBGS_lag1",square((GDPC1_t * -0.013530504) / RTWEXBGS_lag...,models/inflation_pysr/pysr_cpiaucsl_h8_kernel_...
174,8,lgbm_v2,0.000028,"FEDFUNDS_GAP_lag1, GDPC1_t",0.006012279 - (square(square(GDPC1_t / FEDFUND...,models/inflation_pysr/pysr_cpiaucsl_h8_lgbm_v2...
